# SILVER_CID

CID-10 padronizado:
- a descrição perde o código repetido no início (`"A00   Colera"` → `"COLERA"`);
- recebe o **capítulo** da CID-10, derivado da letra e do número do código
  (ex.: `J` = capítulo X, Doenças do aparelho respiratório).

**Origem:** `BRONZE_CID`.

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO SILVER_CID (COD_CID, DESCRICAO_CID, CAPITULO_CID)
SELECT COD_CID, MAX(DESCRICAO_CID), MAX(CAPITULO_CID)
FROM (
    SELECT TRIM(CD_COD) AS COD_CID,
           UPPER(TRIM(REGEXP_REPLACE(CD_DESCR, '^\S+\s+', ''))) AS DESCRICAO_CID,
           CASE
               WHEN SUBSTR(CD_COD, 1, 1) IN ('A', 'B') THEN 'I - DOENCAS INFECCIOSAS E PARASITARIAS'
               WHEN SUBSTR(CD_COD, 1, 1) = 'C'
                 OR SUBSTR(CD_COD, 1, 2) IN ('D0', 'D1', 'D2', 'D3', 'D4') THEN 'II - NEOPLASIAS'
               WHEN SUBSTR(CD_COD, 1, 1) = 'D' THEN 'III - DOENCAS DO SANGUE E IMUNITARIAS'
               WHEN SUBSTR(CD_COD, 1, 1) = 'E' THEN 'IV - DOENCAS ENDOCRINAS E METABOLICAS'
               WHEN SUBSTR(CD_COD, 1, 1) = 'F' THEN 'V - TRANSTORNOS MENTAIS E COMPORTAMENTAIS'
               WHEN SUBSTR(CD_COD, 1, 1) = 'G' THEN 'VI - DOENCAS DO SISTEMA NERVOSO'
               WHEN SUBSTR(CD_COD, 1, 2) IN ('H0', 'H1', 'H2', 'H3', 'H4', 'H5') THEN 'VII - DOENCAS DO OLHO'
               WHEN SUBSTR(CD_COD, 1, 1) = 'H' THEN 'VIII - DOENCAS DO OUVIDO'
               WHEN SUBSTR(CD_COD, 1, 1) = 'I' THEN 'IX - DOENCAS DO APARELHO CIRCULATORIO'
               WHEN SUBSTR(CD_COD, 1, 1) = 'J' THEN 'X - DOENCAS DO APARELHO RESPIRATORIO'
               WHEN SUBSTR(CD_COD, 1, 1) = 'K' THEN 'XI - DOENCAS DO APARELHO DIGESTIVO'
               WHEN SUBSTR(CD_COD, 1, 1) = 'L' THEN 'XII - DOENCAS DA PELE'
               WHEN SUBSTR(CD_COD, 1, 1) = 'M' THEN 'XIII - DOENCAS OSTEOMUSCULARES'
               WHEN SUBSTR(CD_COD, 1, 1) = 'N' THEN 'XIV - DOENCAS DO APARELHO GENITURINARIO'
               WHEN SUBSTR(CD_COD, 1, 1) = 'O' THEN 'XV - GRAVIDEZ, PARTO E PUERPERIO'
               WHEN SUBSTR(CD_COD, 1, 1) = 'P' THEN 'XVI - AFECCOES DO PERIODO PERINATAL'
               WHEN SUBSTR(CD_COD, 1, 1) = 'Q' THEN 'XVII - MALFORMACOES CONGENITAS'
               WHEN SUBSTR(CD_COD, 1, 1) = 'R' THEN 'XVIII - SINTOMAS E ACHADOS ANORMAIS'
               WHEN SUBSTR(CD_COD, 1, 1) IN ('S', 'T') THEN 'XIX - LESOES E ENVENENAMENTOS'
               WHEN SUBSTR(CD_COD, 1, 1) IN ('V', 'W', 'X', 'Y') THEN 'XX - CAUSAS EXTERNAS'
               WHEN SUBSTR(CD_COD, 1, 1) = 'Z' THEN 'XXI - FATORES QUE INFLUENCIAM A SAUDE'
               WHEN SUBSTR(CD_COD, 1, 1) = 'U' THEN 'XXII - CODIGOS ESPECIAIS'
               ELSE 'NAO INFORMADO'
           END AS CAPITULO_CID
    FROM BRONZE_CID
    WHERE TRIM(CD_COD) IS NOT NULL
)
GROUP BY COD_CID
"""

In [ ]:
recarregar("SILVER_CID", SQL, origem="BRONZE_CID")

In [ ]:
consultar("SELECT * FROM SILVER_CID FETCH FIRST 10 ROWS ONLY")